In [19]:
# A quick bunch of import statements that we'll need later:

import pandas as pd
import math
import heapq

In [ ]:
"""
Beginning of Orion's work...
"""

# Problem 1: Urban Route Resilience

For this assignment we were asked to choose a city. I, Orion, while having grown up originally way out on Cape Code, Mass, spent most of my adult life in Boston, MA. I love that city with all my heart. It's my homeland! And so Thomas and I chose it to work with for this assignment. This is partially because of how sentimental I am for it, but also because good lord does it have rush hour traffic. And LOTS of one-way streets. I chose 10 locations in the city that have special meaning for me (places I've lived, places I've worked, all kinds of things!) and placed them onto a satellite map of the city:

<div align="center">
    <img src="1.png" width="1200">
</div>

Next, I drew in the rough impression of the various roads and routes that lead a person from many of these to many others of these. Here's what that diagram looks like:

<div align="center">
    <img src="2.png" width="1200">
</div>

Uhhhhhh oh. So sorry & my bad. I accidentally messed up and chose 11 parts of the city. Whooops! We'll go with 11 locations instead of 10. My apologies.

That being said...

# 1.1: City and Graph Design

I once lived way out West in Watertown. It's kindof a crappy little suburbs, haha. I was working at a restaurant in Harvard Square at the time and taking community college classes there too and it was a fairly cheap little town to commute in from by bus. Let's call that point the starting point, so I'll be named point 'A'. Suppose that a person out in Watertown has a plane ride to catch out at Logan Airport. That can be our destination node. We'll called it 'K'. Let's try and convert this all into a nice, simplified graph version.

<div align="center">
    <img src="3.png" width="1200">
</div>

I've additionally added in a few (slightly imaginary) one-way-only paths colored in black. Just for fun.

Let's now add it some slightly imaginary estimates of how long each of these commutes takes at around 1:30am when there's truly no traffic. Routes from West-to-East will be marked *above* the arrow. Routes from East-to-West will be marked *below* the arrow.

All times represent drive times in minutes. These are the actual real drive times Google Maps had for Boston on Novemeber 23rd at 1:30am Boston-time (11:30pm here in Denver):

<div align="center">
    <img src="4.png" width="1200">
</div>

Lastly, we're requested to provide a table documenting the edges. We're supposed to mark (three?) of the roads as vulnerable? I'm from Boston. Y'all don't get it. *ALL* of the roads are vulnerable! Any road could close 24/7 for any reason. Traffic accident? School lets out? Red Sox game lets out? ... All the roads are vulnerable, across the whole city.

| From | To | Base Time (minutes)| Direction | Source | Vulnerable? |
|---|---|---:|---|---|---|
| A | B | 14 | Two way | Google Maps | Yes |
| B | A | 16 | Two way | Google Maps | Yes |
| A | C | 13 | Two way | Google Maps | Yes |
| C | A | 12 | Two way | Google Maps | Yes |
| A | D | 11 | Two way | Google Maps | Yes |
| D | A | 10 | Two way | Google Maps | Yes |
| A | H | 24 | Two way | Google Maps | Yes |
| H | A | 22 | Two way | Google Maps | Yes |
| B | F | 7 | Two way | Google Maps | Yes |
| F | B | 8 | Two way | Google Maps | Yes |
| C | F | 8 | Two way | Google Maps | Yes |
| F | C | 9 | Two way | Google Maps | Yes |
| C | D | 9 | Two way | Google Maps | Yes |
| D | C | 5 | Two way | Google Maps | Yes |
| D | E | 5 | Two way | Google Maps | Yes |
| E | D | 6 | Two way | Google Maps | Yes |
| E | G | 6 | Two way | Google Maps | Yes |
| G | E | 6 | Two way | Google Maps | Yes |
| F | J | 12 | Two way | Google Maps | Yes |
| J | F | 8 | Two way | Google Maps | Yes |
| G | I | 6 | Two way | Google Maps | Yes |
| I | G | 5 | Two way | Google Maps | Yes |
| I | J | 12 | Two way | Google Maps | Yes |
| J | I | 12 | Two way | Google Maps | Yes |
| J | K | 11 | Two way | Google Maps | Yes |
| K | J | 8 | Two way | Google Maps | Yes |
| H | K | 21 | Two way | Google Maps | Yes |
| K | H | 22 | Two way | Google Maps | Yes |
| C | B | 6 | One way | Google Maps | Yes |
| G | C | 9 | One way | Google Maps | Yes |
| H | G | 8 | One way | Google Maps | Yes |
| I | H | 14 | One way | Google Maps | Yes |

# 1.2: Graph Representation

Let's now make an adjacency matrix to represent this graph! ... Paths that *have* an arrow between them will be marked with the number of minutes that arrow takes to drive. Path that *do not have* an arrow connection will be left empty.

Note that the left axis represents *from* which node while the top axis represents *to* which node.

| From/To | A | B | C | D | E | F | G | H | I | J | K |
|---|---|---|---|---|---|---|---|---|---|---|---|
| A |  | 14 | 13 | 11 | | | | 24 | | | |
| B | 16 | | | | | 7 | | | | | |
| C | 12 | 6 | | 9 | | 8 | | | | | |
| D | 10 | | 5 | | 5 | | | | | | |
| E | | | | 6 | | 6 | | | | | |
| F | | 8 | 9 | | | | | | | 12 | |
| G | | | 9 | | 6 | | | | 6 | | |
| H | 22 | | | | | | 8 | | | | 21 |
| I | | | | | | | 5 | 14 |  12 | | |
| J | | | | | | 8 | | 12 | | | 11 |
| K | | | | | | | | 22 | | 8 | |

Let's now convert the above into some Python code using the Pandas module.

In [20]:
# For nodes that are NOT connected, I'll simply record those as a "_" symbol

matrix_info = [
    ["_", 14, 13, 11,  "_", "_", "_", 24, "_", "_", "_"],
    [16, "_", "_", "_", "_",  7, "_", "_", "_", "_", "_"],
    [12,  6, "_",  9, "_",  8, "_", "_", "_", "_", "_"],
    [10, "_",  5, "_",  5, "_", "_", "_", "_", "_", "_"],
    ["_", "_", "_",  6, "_",  6, "_", "_", "_", "_", "_"],
    ["_",  8,  9, "_", "_", "_", "_", "_", "_", 12, "_"],
    ["_", "_",  9, "_",  6, "_", "_", "_",  6, "_", "_"],
    [22, "_", "_", "_", "_", "_",  8, "_", "_", "_", 21],
    ["_", "_", "_", "_", "_", "_",  5, 14, 12, "_", "_"],
    ["_", "_", "_", "_", "_",  8, "_", 12, "_", "_", 11],
    ["_", "_", "_", "_", "_", "_", "_", 22, "_",  8, "_"]
]

df = pd.DataFrame(matrix_info,
                  index=["A", "B", "C", "D", "E", "F", "G", "H", "I", "J", "K"],
                  columns=["A", "B", "C", "D", "E", "F", "G", "H", "I", "J", "K"])

print(df)

    A   B   C   D  E  F  G   H   I   J   K
A   _  14  13  11  _  _  _  24   _   _   _
B  16   _   _   _  _  7  _   _   _   _   _
C  12   6   _   9  _  8  _   _   _   _   _
D  10   _   5   _  5  _  _   _   _   _   _
E   _   _   _   6  _  6  _   _   _   _   _
F   _   8   9   _  _  _  _   _   _  12   _
G   _   _   9   _  6  _  _   _   6   _   _
H  22   _   _   _  _  _  8   _   _   _  21
I   _   _   _   _  _  _  5  14  12   _   _
J   _   _   _   _  _  8  _  12   _   _  11
K   _   _   _   _  _  _  _  22   _   8   _


In [21]:
## Note to myself for later use (cause I'm still slightly a rookie in Pandas... kinda):

## This is the code to access the weight of the edge from A to B
print(df.loc["A", "B"])

# This is the code to access all the nodes that A connects to
#print(df.loc["A"])

14


The assignment next asks for some explanations. Let's do it.

*Select and justify a graph representation, such as an adjacency list, adjacency matrix, edge list, or a useful combination*.<br>
I chose to use an adjacency matrix. I tend to always like these best, as they're nice and simple and quickly (and easily) display ALL of the info a person could ever want. I lean away from these only in cases where the number of nodes is huge (maybe 60+) such that it would no longer easily display all on the screen at once. In such a case, especially if the edges are sparse and rare, an edge list can be much "kinder" to the human eye. But... If it's just an absolutely unfathomably Cthulhu-like massive matrix that no human could ever read anyway, then either an adjacency list or an edge list is probably the way to go. Since this graph is small, I chose an adjacency matrix.

*How are nodes and weighted directed edges stored*?<br>
Nodes in an adjacency matrix are found as the columns and rows of the matrix. The rows represent what node you're going *from*, the columns represent what node you're going *to*. The weights of the directed arrows between are the elements in the matrix. In short, if you seek to find the number of minutes it'll take to get from node H to node E, just scroll down visually until you're on row H, move across until you're also on column E, and read the number!

*How are one way roads represented?*<br>
One way roads can be spotted in the matrix as any NON-SYMMETRIES in the matrix. Not in the weights of the numbers, but rather in which paths are only found as a number in either the lower left or the upper right of the matrix, while their mirror image is a "_" symbol. Those cases represent times where there *is* a drive time in one direction and simply *isn't* in the other direction.

*How the representation supports Dijkstra’s Algorithm*?<br>
An adjacency matrix supports Dijkstra's Algorithm by housing all the info it needs to function. Once the algorithm arrives to a particular node, it can simply scan horizontally across that row to find all the node connections (the numbers) and skip over all the node non-connections (the "_" elements). The Algoritm can be set up to fully function using only the matrix to work!

*How do its memory requirements compare with an adjacency list or an edge list*?<br>
These are just three known, studied, to-be-memorized values that a data science student eventually learns.<br>
<br>
An adjacency matrix has storage needs of $O(V^2)$, where $V$ is the number of nodes. This is because the matrix has an info slot for every single theoretical vertex-to-vertex connection that *might* exist, even if they don't actually have a connection (the "_" elements in my above matrix).<br>
<br>
An adjaceny list has storage needs of $O(V + E)$, where $E$ is the number of edges. That's the primary other reason if the matrix were HUGE it might be wise to not use an adjacency matrix. It stores all the info, but can be quite memory hungry. We don't need to worry about that here, with such a small little matrix.<br>
<br>
Lastly an edge list has storage needs of just $O(E)$. It simply focuses on the info of the edges. All the same info can be represented there, and for a MASSIVE matrix, that might be ideal. Especially a sparse one. But it's much much harder for the human eye to read that and parse it into some vague sense of what the graph looks like. That's why I like adjacency matrices. Then again, none of these are even half as good as a good-old plotted drawing of the graph. *THAT'S* what a human eye can make sense of.

# 1.3: Dijkstra’s Algorithm

In [22]:
numbers = [14, 3, 27, 8, 1]
heapq.heapify(numbers)
print(numbers)
heapq.heappush(numbers, -1)
print(numbers)
a = heapq.heappop(numbers)
print(numbers)
print(a)

[1, 3, 27, 8, 14]
[-1, 3, 1, 8, 14, 27]
[1, 3, 27, 8, 14]
-1


In [27]:
## Let's now implement Dijkstra's Algorithm in code. I'll design it to be able to compute the shortest distance and
## shortest route from any point to any other point.

## I'll try and make it be capable of running from inputs of any adjacency matrix, any starting point, and any end point.

def Dijkstra(matrix, start, end):

    ## Begin with some starting info, almost like a Class object's __init__
    
    # Gather the names of all the nodes
    nodes = list(matrix.index)
    
    # Build our dictionary of shortest known paths
    shorted_known_paths = {}
    for node in nodes:
        if node == start:
            shorted_known_paths[node] = (0, False)
               ## This is (distance, previous node)
        else:
            shorted_known_paths[node] = (math.inf, False)

    # Get the priority queue up and running (I'll call it pq for short)
    pq = [(0, start)]

    # This is the basic core loop
    while len(pq) > 0:
        
        # Pop the non-visited node that has the closest so-far-discovered distance
        dist, current_node = heapq.heappop(pq)
        # A not-yet-visited node might appear on the heap TWICE!
        # If we've already found a better way to get there, we should skip the less-optimal route
        if dist > shorted_known_paths[current_node][0]:
            continue

        # Which nodes does that one connect to?
        connections = []
        for node in nodes:
            possible_connection_weight = matrix.loc[current_node, node]
            if possible_connection_weight != "_":
                connections.append((possible_connection_weight, (node)))

        # Now update the shorted_known_paths dict
        for connection in connections:
            # If the new way to get there is less than the current one in the dict, update it
            if dist + connection[0] < shorted_known_paths[connection[1]][0]:
                shorted_known_paths[connection[1]] = (dist + connection[0], current_node)
                heapq.heappush(pq, (dist + connection[0], connection[1]))

        # If we've found the distance to the desired node, it's time to stop
        if current_node == end:
            break

    # Well... We did all of the exploring. Now we need to package and return the results
    #
    # I want to return, at least, the shortest distance found and what path you take to achieve it!
    shortest_dist = shorted_known_paths[end][0]

    # Did we ever actually find a way to get there?
    # If not...
    if shortest_dist == math.inf:
        return shortest_dist, False

    # If so...
    the_path = [end]
    cursor = end
    while cursor != False:
        the_path.append(shorted_known_paths[cursor][1])
        cursor = shorted_known_paths[cursor][1]
    the_path.reverse()
    return shortest_dist, the_path


# Note! If there's a path from start to end, it'll return the distance and the path
#       If not, it'll return (math.inf, False)

Whew! That was a lot of work. Let's test it out!

In [28]:
print(Dijkstra(df, "A", "K"))

(44, [False, 'A', 'C', 'F', 'J', 'K'])


In [ ]:
## SIIIIICKKKK!!!! It seems to work!

In [ ]:
"""
End of Orion's work (for now)
"""

# Saved for later, the rush hour version

And let's do another graph of this in rush hour. I once again went to Google Maps and looked up the drive times, but this time it was the estimated time for Tuesday at 5:15pm. Let's take a look at what the rush hour graphs look like:

<div align="center">
    <img src="5.png" width="1200">
</div>